In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_colwidth", 800)

### Review the workload context and usage assumptions provided in the notebook

In [2]:
# ---------------------------------------------------------
# Workload context and usage assumptions
# ---------------------------------------------------------
workload_context = {
    "workload_name": "Customer-facing GenAI shopping assistant (interactive chat)",
    "traffic_pattern": "Steady daytime traffic with evening peaks (~2x median); interactive, latency-sensitive",
    "requests_per_day": 120_000,
    "avg_input_tokens": 650,
    "avg_output_tokens": 220,
    "sla_notes": "Interactive SLA: p95 end-to-end latency under 1200 ms; availability 99.9%",
    "kg_co2e_per_kwh": 0.38,  # grid-average emission factor assumption
}

display(pd.DataFrame([workload_context]))

,workload_name,traffic_pattern,requests_per_day,avg_input_tokens,avg_output_tokens,sla_notes,kg_co2e_per_kwh
0,Customer-facing GenAI shopping assistant (interactive chat),"Steady daytime traffic with evening peaks (~2x median); interactive, latency-sensitive",120000,650,220,Interactive SLA: p95 end-to-end latency under 1200 ms; availability 99.9%,0.38


### Load the baseline inference metrics for the simulated generative AI workload

In [3]:
# ---------------------------------------------------------
# Load the baseline inference metrics
# ---------------------------------------------------------
baseline_path = Path("baseline_inference_metrics.csv")

if not baseline_path.exists():
    raise FileNotFoundError(
        f"Expected {baseline_path.resolve()} - make sure you run the notebook "
        "from the starter/ directory containing baseline_inference_metrics.csv")

baseline_df = pd.read_csv(baseline_path)
display(baseline_df)

,metric,baseline_value,notes
0,p50_latency_ms,420.0,Median end-to-end inference latency
1,p95_latency_ms,980.0,Tail latency under moderate load
2,throughput_requests_per_sec,18.0,Sustained throughput under steady traffic
3,average_memory_gb,22.5,Average GPU memory utilization
4,cost_per_1k_requests_usd,7.4,Estimated cloud cost based on instance pricing
5,estimated_energy_kwh_per_1k_requests,3.2,Rough estimate based on compute utilization


### Analyze the baseline metrics to identify potential efficiency concerns

In [4]:
# Pull baseline values into a dict for easy downstream calculations.
baseline = dict(zip(baseline_df["metric"], baseline_df["baseline_value"]))

# A few simple checks based on the workload notes.
baseline_findings = []

p95 = baseline.get("p95_latency_ms")
if p95 is not None:
    if p95 > 1200:
        baseline_findings.append(f"p95 latency is {p95:.0f} ms, which may be high for an interactive SLA.")
    else:
        baseline_findings.append(f"p95 latency is {p95:.0f} ms, within the stated interactive target range.")

mem = baseline.get("average_memory_gb")
if mem is not None:
    if mem >= 24:
        baseline_findings.append(f"Average memory is {mem:.1f} GB, which may force larger (more expensive) GPUs.")
    elif mem >= 16:
        baseline_findings.append(f"Average memory is {mem:.1f} GB; watch GPU sizing and headroom for spikes.")
    else:
        baseline_findings.append(f"Average memory is {mem:.1f} GB, leaving reasonable headroom for common GPU tiers.")

cost = baseline.get("cost_per_1k_requests_usd")
if cost is not None:
    baseline_findings.append(f"Estimated cost is ${cost:.2f} per 1k requests. Scale impact depends on daily volume.")

energy = baseline.get("estimated_energy_kwh_per_1k_requests")
if energy is not None:
    baseline_findings.append(f"Estimated energy is {energy:.2f} kWh per 1k requests. Consider energy as a constraint at scale.")

pd.DataFrame({"baseline_findings": baseline_findings})

,baseline_findings
0,"p95 latency is 980 ms, within the stated interactive target range."
1,Average memory is 22.5 GB; watch GPU sizing and headroom for spikes.
2,Estimated cost is $7.40 per 1k requests. Scale impact depends on daily volume.
3,Estimated energy is 3.20 kWh per 1k requests. Consider energy as a constraint at scale.


### Review the simulated optimization scenario and its post-optimization metrics

#### Optimization Scenario 

For this exercise, assume the following optimization has already been applied to the generative AI inference system.

##### Optimization Applied
**INT8 quantization with dynamic batching**

##### Description
- Reduced-precision (INT8) inference is used to lower memory usage and improve throughput.
- Dynamic batching groups multiple requests together to increase hardware utilization under steady traffic.

##### Expected Benefits
- Lower GPU memory footprint
- Higher sustained throughput
- Reduced cost per request
- Reduced energy consumption per request

##### Known Tradeoffs
- Median latency may increase slightly due to batching overhead
- Output quality impact is expected to be minor but not zero
- Occasional formatting drift or subtle tone changes may occur for longer outputs

##### Your Task
You are **not** asked to design or implement this optimization.

Your task is to **evaluate whether this optimization is acceptable** for the workload using the provided performance, cost, and energy metrics.


In [5]:
optimization_scenario = {
    "name": "INT8 quantization + dynamic batching",
    "notes": [
        "Reduced precision inference lowers memory and can improve throughput.",
        "Batching can increase throughput but may increase median latency under certain traffic patterns.",
        "Quality impact is assumed to be minimal but not zero (monitor formatting + factuality).",
    ],
    "quality_impact_note": "Minor: occasional slight tone shift and rare formatting drift under long outputs.",
}

pd.DataFrame({"scenario_notes": optimization_scenario["notes"]})

,scenario_notes
0,Reduced precision inference lowers memory and can improve throughput.
1,Batching can increase throughput but may increase median latency under certain traffic patterns.
2,Quality impact is assumed to be minimal but not zero (monitor formatting + factuality).


### Construct a comparison table that summarizes baseline and optimized metrics side by side

In [6]:
# ---------------------------------------------------------
# Post-optimization metrics (INT8 quantization + dynamic batching)
# ---------------------------------------------------------
# Values represent the simulated measurements after applying the optimization
# scenario described above: memory and cost drop sharply with quantization,
# throughput rises with batching, median latency rises slightly (batching
# overhead) while tail latency improves (less queueing at peak).
optimized_metrics = {
    "p50_latency_ms": 460.0,                       # up slightly: batching overhead
    "p95_latency_ms": 840.0,                       # down: less queueing under load
    "throughput_requests_per_sec": 31.0,           # up: INT8 + batched execution
    "average_memory_gb": 12.8,                     # down: 8-bit weights/activations
    "cost_per_1k_requests_usd": 4.30,              # down: smaller instance class
    "estimated_energy_kwh_per_1k_requests": 1.9,   # down: higher utilization efficiency
}

optimized_df = pd.DataFrame({
    "metric": list(optimized_metrics.keys()),
    "optimized_value": list(optimized_metrics.values()),
})

comparison = baseline_df.merge(optimized_df, on="metric", how="left")
comparison["delta"] = comparison["optimized_value"] - comparison["baseline_value"]
comparison["pct_change"] = (comparison["delta"] / comparison["baseline_value"] * 100).round(1)
display(comparison)

,metric,baseline_value,notes,optimized_value,delta,pct_change
0,p50_latency_ms,420.0,Median end-to-end inference latency,460.0,40.0,9.5
1,p95_latency_ms,980.0,Tail latency under moderate load,840.0,-140.0,-14.3
2,throughput_requests_per_sec,18.0,Sustained throughput under steady traffic,31.0,13.0,72.2
3,average_memory_gb,22.5,Average GPU memory utilization,12.8,-9.7,-43.1
4,cost_per_1k_requests_usd,7.4,Estimated cloud cost based on instance pricing,4.3,-3.1,-41.9
5,estimated_energy_kwh_per_1k_requests,3.2,Rough estimate based on compute utilization,1.9,-1.3,-40.6


### Calculate and summarize the differences between baseline and optimized metrics

In [7]:
requests_per_day = workload_context["requests_per_day"]

# Pull per-1k request cost/energy and scale them
baseline_cost_per_day = (baseline["cost_per_1k_requests_usd"] * requests_per_day) / 1000
optimized_cost_per_day = (optimized_metrics["cost_per_1k_requests_usd"] * requests_per_day) / 1000

baseline_energy_per_day = (baseline["estimated_energy_kwh_per_1k_requests"] * requests_per_day) / 1000
optimized_energy_per_day = (optimized_metrics["estimated_energy_kwh_per_1k_requests"] * requests_per_day) / 1000

# Optional CO2e estimate (assumption-based)
kg_co2e_per_kwh = workload_context.get("kg_co2e_per_kwh", None)
baseline_co2e_kg_per_day = baseline_energy_per_day * kg_co2e_per_kwh if kg_co2e_per_kwh is not None else None
optimized_co2e_kg_per_day = optimized_energy_per_day * kg_co2e_per_kwh if kg_co2e_per_kwh is not None else None

scale_summary = {
    "requests_per_day": requests_per_day,
    "baseline_cost_per_day_usd": baseline_cost_per_day,
    "optimized_cost_per_day_usd": optimized_cost_per_day,
    "daily_cost_savings_usd": baseline_cost_per_day - optimized_cost_per_day,
    "baseline_energy_kwh_per_day": baseline_energy_per_day,
    "optimized_energy_kwh_per_day": optimized_energy_per_day,
    "daily_energy_savings_kwh": baseline_energy_per_day - optimized_energy_per_day,
}

if baseline_co2e_kg_per_day is not None:
    scale_summary.update({
        "assumed_kg_co2e_per_kwh": kg_co2e_per_kwh,
        "baseline_co2e_kg_per_day": baseline_co2e_kg_per_day,
        "optimized_co2e_kg_per_day": optimized_co2e_kg_per_day,
        "daily_co2e_savings_kg": baseline_co2e_kg_per_day - optimized_co2e_kg_per_day,
    })

pd.DataFrame([scale_summary]).T.rename(columns={0: "value"})

,value
requests_per_day,120000.00
baseline_cost_per_day_usd,888.00
optimized_cost_per_day_usd,516.00
daily_cost_savings_usd,372.00
baseline_energy_kwh_per_day,384.00
optimized_energy_kwh_per_day,228.00
daily_energy_savings_kwh,156.00
assumed_kg_co2e_per_kwh,0.38
baseline_co2e_kg_per_day,145.92
optimized_co2e_kg_per_day,86.64


### Evaluate the tradeoffs introduced by the optimization

In [8]:
tradeoffs = []

# Latency tradeoff (p50 may increase; p95 may improve)
tradeoffs.append({
    "area": "Latency",
    "what_changed": f"p50: {baseline['p50_latency_ms']} -> {optimized_metrics['p50_latency_ms']} ms, "
                    f"p95: {baseline['p95_latency_ms']} -> {optimized_metrics['p95_latency_ms']} ms",
    "interpretation": "Median latency slightly increased (batching overhead), but tail latency improved."
})

# Throughput
tradeoffs.append({
    "area": "Throughput",
    "what_changed": f"{baseline['throughput_requests_per_sec']} -> {optimized_metrics['throughput_requests_per_sec']} req/s",
    "interpretation": "Higher throughput reduces queueing risk and can stabilize tail latency under load."
})

# Memory
tradeoffs.append({
    "area": "Memory",
    "what_changed": f"{baseline['average_memory_gb']} -> {optimized_metrics['average_memory_gb']} GB",
    "interpretation": "Lower memory enables smaller instances or more headroom, improving cost flexibility."
})

# Quality / flexibility (scenario-based note)
tradeoffs.append({
    "area": "Output Quality",
    "what_changed": optimization_scenario["quality_impact_note"],
    "interpretation": "Treat as a monitoring requirement. Add regression tests for tone, formatting, and factuality."
})

pd.DataFrame(tradeoffs)

,area,what_changed,interpretation
0,Latency,"p50: 420.0 -> 460.0 ms, p95: 980.0 -> 840.0 ms","Median latency slightly increased (batching overhead), but tail latency improved."
1,Throughput,18.0 -> 31.0 req/s,Higher throughput reduces queueing risk and can stabilize tail latency under load.
2,Memory,22.5 -> 12.8 GB,"Lower memory enables smaller instances or more headroom, improving cost flexibility."
3,Output Quality,Minor: occasional slight tone shift and rare formatting drift under long outputs.,"Treat as a monitoring requirement. Add regression tests for tone, formatting, and factuality."


### Short analysis and clear recommendation 

In [9]:
analysis = f"""
EFFICIENCY ANALYSIS - INT8 QUANTIZATION + DYNAMIC BATCHING
==========================================================
Biggest improvements
- Cost: ${baseline['cost_per_1k_requests_usd']:.2f} -> ${optimized_metrics['cost_per_1k_requests_usd']:.2f}
  per 1k requests (-{(1 - optimized_metrics['cost_per_1k_requests_usd']/baseline['cost_per_1k_requests_usd'])*100:.0f}%),
  or ~${(baseline['cost_per_1k_requests_usd'] - optimized_metrics['cost_per_1k_requests_usd']) * workload_context['requests_per_day'] / 1000:,.0f}/day
  (~${(baseline['cost_per_1k_requests_usd'] - optimized_metrics['cost_per_1k_requests_usd']) * workload_context['requests_per_day'] / 1000 * 365:,.0f}/year) at {workload_context['requests_per_day']:,} requests/day.
- Energy: {baseline['estimated_energy_kwh_per_1k_requests']:.1f} -> {optimized_metrics['estimated_energy_kwh_per_1k_requests']:.1f} kWh
  per 1k requests (-{(1 - optimized_metrics['estimated_energy_kwh_per_1k_requests']/baseline['estimated_energy_kwh_per_1k_requests'])*100:.0f}%), cutting
  ~{(baseline['estimated_energy_kwh_per_1k_requests'] - optimized_metrics['estimated_energy_kwh_per_1k_requests']) * workload_context['requests_per_day'] / 1000 * workload_context['kg_co2e_per_kwh']:,.0f} kg CO2e/day at the assumed {workload_context['kg_co2e_per_kwh']} kg/kWh grid factor.
- Throughput: {baseline['throughput_requests_per_sec']:.0f} -> {optimized_metrics['throughput_requests_per_sec']:.0f} req/s (+{(optimized_metrics['throughput_requests_per_sec']/baseline['throughput_requests_per_sec'] - 1)*100:.0f}%),
  which comfortably covers the ~2x evening peak with headroom.
- Memory: {baseline['average_memory_gb']:.1f} -> {optimized_metrics['average_memory_gb']:.1f} GB (-{(1 - optimized_metrics['average_memory_gb']/baseline['average_memory_gb'])*100:.0f}%), unlocking smaller GPU tiers.
- Tail latency: p95 {baseline['p95_latency_ms']:.0f} -> {optimized_metrics['p95_latency_ms']:.0f} ms, further inside the 1200 ms SLA.

Biggest tradeoffs
- Median latency rises {baseline['p50_latency_ms']:.0f} -> {optimized_metrics['p50_latency_ms']:.0f} ms (+{(optimized_metrics['p50_latency_ms']/baseline['p50_latency_ms'] - 1)*100:.0f}%) from batching overhead -
  perceptible but well within the interactive SLA.
- Quality impact is 'minor but not zero': INT8 can cause occasional tone shift
  and formatting drift on long outputs, so it must be monitored, not assumed away.
- Batching ties efficiency to steady traffic; very low-traffic windows see less benefit.

Acceptability
For this workload - interactive but with a 1200 ms p95 SLA, high daily volume, and
cost/energy pressure at scale - the optimization is acceptable: every SLA-relevant
metric stays inside target, and the cost/energy wins are large and recurring. The
residual risk is output quality, which is manageable with regression monitoring.
""".strip()
print(analysis)

EFFICIENCY ANALYSIS - INT8 QUANTIZATION + DYNAMIC BATCHING
Biggest improvements
- Cost: $7.40 -> $4.30
  per 1k requests (-42%),
  or ~$372/day
  (~$135,780/year) at 120,000 requests/day.
- Energy: 3.2 -> 1.9 kWh
  per 1k requests (-41%), cutting
  ~59 kg CO2e/day at the assumed 0.38 kg/kWh grid factor.
- Throughput: 18 -> 31 req/s (+72%),
  which comfortably covers the ~2x evening peak with headroom.
- Memory: 22.5 -> 12.8 GB (-43%), unlocking smaller GPU tiers.
- Tail latency: p95 980 -> 840 ms, further inside the 1200 ms SLA.

Biggest tradeoffs
- Median latency rises 420 -> 460 ms (+10%) from batching overhead -
  perceptible but well within the interactive SLA.
- Quality impact is 'minor but not zero': INT8 can cause occasional tone shift
  and formatting drift on long outputs, so it must be monitored, not assumed away.
- Batching ties efficiency to steady traffic; very low-traffic windows see less benefit.

Acceptability
For this workload - interactive but with a 1200 ms p95 SLA, 

In [10]:
# ---------------------------------------------------------
# Recommendation
# ---------------------------------------------------------
recommendation = {
    "decision": "Adopt with guardrails",
    "guardrails": [
        "Output-quality regression suite (tone, formatting, factuality) run on every build; block rollout on regression beyond agreed thresholds",
        "Canary deployment: 10% traffic for 2 weeks with side-by-side quality sampling vs FP16 baseline",
        "Latency SLO alerting on p50 > 600 ms and p95 > 1100 ms with automatic batch-size fallback",
        "Quarterly review of batching efficiency against traffic pattern drift",
    ],
    "why": (
        "All SLA-relevant metrics remain inside target (p95 840 ms vs 1200 ms SLA) while "
        "cost drops ~42% (~$135k/year at current volume), energy drops ~41%, and throughput "
        "rises ~72%, covering peak traffic with headroom. The only regressions - +40 ms median "
        "latency and minor, monitorable quality risk from INT8 - are small and controllable, "
        "so adoption is justified with quality and latency guardrails in place."
    ),
}

display(pd.DataFrame({
    "decision": [recommendation["decision"]],
    "why": [recommendation["why"]],
    "guardrails": ["; ".join(recommendation.get("guardrails", []))]
}))

,decision,why,guardrails
0,Adopt with guardrails,"All SLA-relevant metrics remain inside target (p95 840 ms vs 1200 ms SLA) while cost drops ~42% (~$135k/year at current volume), energy drops ~41%, and throughput rises ~72%, covering peak traffic with headroom. The only regressions - +40 ms median latency and minor, monitorable quality risk from INT8 - are small and controllable, so adoption is justified with quality and latency guardrails in place.","Output-quality regression suite (tone, formatting, factuality) run on every build; block rollout on regression beyond agreed thresholds; Canary deployment: 10% traffic for 2 weeks with side-by-side quality sampling vs FP16 baseline; Latency SLO alerting on p50 > 600 ms and p95 > 1100 ms with automatic batch-size fallback; Quarterly review of batching efficiency against traffic pattern drift"
